# ZuCo paper methods: Python engineering pilot

**CPU → Run all.** This tests one original SR block (ZAB, block 1), with all 105 scalp channels.

**This is not yet the paper reproduction or a sentiment comparison.** It validates trigger synchronization, continuous nuisance modeling and output bookkeeping. No sentiment labels are used. The full word-label join, trial matching, FRP inference and decoding remain subsequent stages.

The code reads the released preprocessed continuous EEG and corrected eye-event file (about 51 MB total) into temporary Colab storage. It does not use or change our extracted trial archive/features. No MATLAB/license is needed.

Python choices are recorded explicitly: centered Hamming FIR with 1/50-Hz cutoffs, 125-Hz resampling, cubic quantile B-splines, and a sparse least-squares model. These have **not** been validated as numerically identical to EEGLAB/Unfold. Released corpus preprocessing is inherited, not a rerun of the paper's Automagic/MARA pipeline.

Outputs should be **below 5 MB**: event timeline, synchronization/solver diagnostics, nuisance coefficients and two all-fixation mean waveforms. Full continuous/individual-epoch signals are not saved. Completed pilot outputs are reused after checksum validation.


In [ ]:
from google.colab import drive
from pathlib import Path
import os, subprocess, sys, json
from IPython.display import display, Image, Markdown

drive.mount('/content/drive')
THESIS=Path('/content/drive/MyDrive/Thesis')
REPO=Path('/content/word-sentiment-eeg-paper')
if not REPO.exists():
    subprocess.run(['git','-c','credential.helper=','-c','http.extraheader=','clone','--depth','1',
                    'https://github.com/parmisbathayan/word-sentiment-eeg.git',str(REPO)],check=True,
                   env=dict(os.environ,GIT_TERMINAL_PROMPT='0'))
else:
    if subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True).strip():
        raise RuntimeError('Temporary checkout has edits; preserve them before updating.')
    subprocess.run(['git','-C',str(REPO),'pull','--ff-only','origin','main'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(REPO/'requirements-analysis.txt')],check=True)
OUT=THESIS/'Results/word_sentiment_eeg/paper_python_pilot/zuco/ZAB_SR1'
print('CPU only. Temporary cloud download: approximately 51 MB; estimated Drive outputs below 5 MB.')


In [ ]:
sys.path.insert(0,str(REPO/'scripts'))
from colab_progress import run_with_progress
command=[sys.executable,'-u',str(REPO/'scripts/zuco_paper_pilot.py'),'--output',str(OUT),'--channels','105']
run_with_progress(command,OUT/'runtime_status.json',activity='ZuCo Python methods engineering pilot')


## Timing, event counts and numerical checks

A successful fit is a numerical check only. It does not establish that the removed activity is exclusively visual/motor activity.


In [ ]:
result=json.loads((OUT/'manifest.json').read_text())
assert result['status']=='completed',result
report=json.loads((OUT/'diagnostics.json').read_text())
display(report['synchronization'])
display(report['event_counts'])
display(report['qa_epoch_counts'])
print('Converged channels:',len(report['solver']))
display(Markdown('### Recorded limitations'))
display(json.loads((OUT/'protocol.json').read_text())['limitations'])


## All-fixation QA comparison

These curves pool fixations without sentiment or word-bound selection. They are not positive/negative FRPs. No epoch baseline subtraction is applied.


In [ ]:
display(Image(filename=str(OUT/'engineering_qa.png')))
